In [ ]:
import os
import re
import time
import pymupdf4llm
import psycopg2
import pandas as pd
from typing import List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel, Field
from pgvector.psycopg2 import register_vector
from google import genai
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
SENIORITY_CONCURRENCY = 10
INDUSTRY_CONCURRENCY = 8
TOP_INDUSTRIES = 3
EMBED_MAX_RETRIES = 3

JOB_POSTINGS_TABLE_NAME = "job_postings"
DB_NAME = "market_fit"
DB_HOST = "localhost"
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
api_key = os.getenv('GEMINI_API_KEY')


def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
    )
    register_vector(conn)
    return conn

def _embed_batch_with_retry(
    client: genai.Client,
    texts: list[str],
    max_retries: int = EMBED_MAX_RETRIES,
) -> list[list[float]]:
    """Embed a batch of strings, retrying on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(model=EMBEDDING_MODEL, contents=texts)
            return [e.values for e in response.embeddings]
        except genai.errors.ClientError as exc:
            is_rate_limit = exc.code == 429 or "RESOURCE_EXHAUSTED" in str(exc.status)
            if not is_rate_limit or attempt == max_retries - 1:
                raise
            match = re.search(r"retry in (\d+(?:\.\d+)?)s", str(exc))
            wait = float(match.group(1)) if match else 2 ** attempt * 10
            print(f"Rate limited — waiting {wait:.0f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
    raise RuntimeError("Max retries exceeded")

def embed_texts(
    client: genai.Client,
    texts: list[str],
    batch_size: int = EMBED_BATCH_SIZE,
    concurrency: int = EMBED_CONCURRENCY,
) -> list[list[float]]:
    """Embed an arbitrary list of texts in parallel batches."""
    batches = [texts[i : i + batch_size] for i in range(0, len(texts), batch_size)]
    results: list[list[list[float]]] = [None] * len(batches)  # type: ignore[list-item]

    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = {pool.submit(_embed_batch_with_retry, client, b): i for i, b in enumerate(batches)}
        for future in as_completed(futures):
            results[futures[future]] = future.result()

    return [emb for batch in results for emb in batch]

def get_job_titles_list() -> List[str]:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title FROM public.{JOB_POSTINGS_TABLE_NAME}")
            rows = cur.fetchall()
            rows = pd.DataFrame(rows, columns=["id", "title"])
    return rows

def update_job_postings_table_with_embeddings(df: pd.DataFrame) -> None:
    with db_connect() as conn:
        with conn.cursor() as cur:
            for _, row in df.iterrows():
                cur.execute(f"""
                    UPDATE {JOB_POSTINGS_TABLE_NAME}
                    SET title_embedding = %s
                    WHERE id = %s
                """, (row["title_embedding"], row["id"]))
            conn.commit()


In [ ]:
client = genai.Client(api_key=api_key)
job_titles_df = get_job_titles_list()
job_titles_df["title_embedding"] = embed_texts(client, job_titles_df["title"].tolist())
update_job_postings_table_with_embeddings(job_titles_df)


,id,title
0,2124334619,Data Engineer | SR AWS / Spark (Remote)
1,2124334408,Graduate Data Engineer
2,2123771365,Data Platform Engineer Senior
3,2123771286,Data Engineer
4,2123414897,Senior Data Engineer (Databricks & Azure)
...,...,...
293,2028612640,Data Engineer (Medior Analyst)
294,2028612639,Senior Data Scientist / Machine Learning Engineer
295,2117628882,Data Engineer
296,2025382368,Data Engineer Pleno
